In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")
huggingface_api_key = os.getenv("HUGGINGFACE_API_KEY")
os.environ["HF_TOKEN"] = huggingface_api_key

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [19]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", convert_system_message_to_human=True)

In [9]:
res = model.invoke("What is the capital of France?")

In [12]:
res

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ce0b1-8b77-7f91-9575-d82460fee106-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

In [13]:
res.content

'The capital of France is **Paris**.'

In [20]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [30]:
# output_parser.parse(model.invoke("What is the capital of china?"))
output_parser.invoke(res)

'The capital of France is **Paris**.'

In [22]:
import warnings
warnings.filterwarnings("ignore")

In [25]:
# Starts from here
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# tell about them 
#  1. InMemoryChatMessageHistory: This class provides an in-memory implementation of a chat message history. It allows you to store and retrieve messages in memory during the runtime of your application. This is useful for scenarios where you want to maintain a conversation history without persisting it to a database or external storage.
#  2. BaseChatMessageHistory: This is an abstract base class that defines the interface for chat message history implementations. It provides a blueprint for how chat message histories should be structured and accessed. You can create custom implementations of this class to suit specific storage needs, such as using a database or file system.
#  3. RunnableWithMessageHistory: This class is designed to be a runnable component that incorporates message history functionality. It allows you to create a runnable that can access and manipulate chat message history, making it easier to build conversational agents that can remember past interactions and context.

In [26]:
from langchain_core.messages import HumanMessage, AIMessage

In [27]:
result = model.invoke([
    HumanMessage(content="Hi my name is BOSS"),
    AIMessage(content="Hello BOSS, how can I help you?"),
    HumanMessage(content="What is my name?")    
])

In [33]:
output_parser.invoke(result)
# Now it is remembering the context of the conversation and giving the correct answer.

'Your name is BOSS.'

In [34]:
store={}

In [36]:
def get_session_chat_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [39]:
model_with_memory = RunnableWithMessageHistory(model, get_session_chat_history)

In [41]:
config = {"configurable": {"session_id": "session-123"}}        

In [42]:
model_with_memory.invoke(HumanMessage(content="Hi my name is BOSS"), config=config).content

"Hi BOSS, it's nice to meet you! How can I help you today?"

In [44]:
model_with_memory.invoke(HumanMessage(content="what is my name?"), config=config).content

'Your name is BOSS.'

In [45]:
config2 = {"configurable": {"session_id": "session-456"}}

In [46]:
model_with_memory.invoke(HumanMessage(content="Have I told you my name?"), config=config2).content

"As a large language model, I don't have memory of past conversations. Therefore, I don't know your name. You haven't told me your name in this interaction."

It means the chatbot can remember the conversation history for each session, allowing it to provide more contextually relevant responses. Each session's chat history is stored in an in-memory data structure, and the chatbot can retrieve this history when needed to generate responses that are informed by previous interactions.

In [48]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [52]:
prompt = ChatPromptTemplate.from_messages(
    [ 
    ("system", "You are a helpful assistant. Give precise and to the point answers."),
    MessagesPlaceholder(variable_name="message")
    ]
)

In [62]:
chatbot_chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_chat_history,
    input_messages_key="message"
)

In [54]:
config

{'configurable': {'session_id': 'session-123'}}

In [69]:
response = chatbot_chain.invoke(
	{"message": [HumanMessage(content="What is my name?")]},
	config=config
)
response.content

AIMessage(content='Your name is BOSS.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ce127-6f19-7633-9867-f08f3bb7e1f8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 5, 'total_tokens': 143, 'input_token_details': {'cache_read': 0}})

Creating a function `get_response` that takes a user query and a session ID, retrieves the chat history for that session, and generates a response using the chatbot chain. The chatbot chain is configured to use the session's chat history when generating responses.

Finally, we have a simple loop that allows the user to interact with the chatbot. The user can type messages, and the chatbot will respond based on the conversation history. The loop continues until the user types "exit" or "quit".

In [64]:
def get_response(query:str, session_id:str):
    config = {"configurable": {"session_id": session_id}}
    response = chatbot_chain.invoke(
        {"message": [HumanMessage(content=query)]},
        config=config
    )
    return response.content

In [66]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the chatbot. Goodbye!")
        break
    bot_response = get_response(user_input, session_id="session-123")
    print(f"Bot: {bot_response}")

Exiting the chatbot. Goodbye!


We can trim the chat history to a certain number of recent messages to manage memory usage while still maintaining enough context for meaningful interactions.